# Gradio UI Testing - Q&A System V2

Quick prototyping notebook for testing the Gradio interface with the refactored pipeline.

In [ ]:
# Setup - adjust paths as needed
import sys
sys.path.append('..')

import gradio as gr
from qanda_module.config import setup_system
from qanda_module.retrieval_lean import process_question_with_style
import logging

# Enable logging to see what's happening
logging.basicConfig(level=logging.INFO)

print("📦 Imports successful!")

In [ ]:
# Setup system - this should work from your previous testing
print("🚀 Setting up system...")

helpers, qa_chain, config = setup_system("../config.yaml")

print("✅ System setup complete!")
print(f"Model: {config.chat_model_name}")
print(f"Temperature: {config.temperature}")
print(f"Debug: {config.debug_enabled}")

In [ ]:
# Test the basic pipeline first (without UI)
print("🧪 Testing basic pipeline...")

test_question = "What did panelists say about climate change?"
print(f"Question: {test_question}")

try:
    result = process_question_with_style(
        helpers=helpers,
        qa_chain=qa_chain,
        query=test_question,
        k=10,  # Small number for testing
        style="Concise",
        config=config
    )
    
    print("✅ Pipeline test successful!")
    print(f"Answer length: {len(result[0])} characters")
    print(f"Status: {result[2]}")
    print(f"\nFirst 200 chars of response:\n{result[0][:200]}...")
    
except Exception as e:
    print(f"❌ Pipeline test failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Define the question handler function
def handle_question(question, k_value, style):
    """Handle user question and return response."""
    if not question.strip():
        return "Please enter a question.", "", "⚠️ No question provided"
    
    print(f"Processing: {question[:50]}...")  # Debug print
    
    try:
        result = process_question_with_style(
            helpers=helpers,
            qa_chain=qa_chain, 
            query=question,
            k=k_value,
            style=style,
            config=config
        )
        
        # result = (formatted_answer, sources, status, pdf_output, tech_info)
        return result[0], result[1], result[2]
        
    except Exception as e:
        error_msg = f"Error: {str(e)}"
        print(f"❌ Error in handler: {error_msg}")
        return error_msg, "", f"❌ {error_msg}"

print("✅ Handler function defined")

In [17]:
def generate_sample_questions(panelist=None, topic=None):
    """Generate comprehensive sample questions."""
    questions = []
    
    # Specific combinations first
    if panelist and topic:
        if len(panelist) == 1:
            questions.extend([
                f"What are {panelist}'s views on {topic.lower()}?",
                f"How did {panelist} respond to questions about {topic.lower()}?",
            ])
        else:
            panelist_names = " and ".join(panelist[:2])  # Show first 2
            questions.extend([
                f"What did {panelist_names} say about {topic.lower()}?",
                f"How do {panelist_names} differ on {topic.lower()}?",
                f"What are the different perspectives from {panelist_names} on {topic.lower()}?",
            ])
    
    # Just panelist questions
    if panelist:
        questions.extend([
            f"What did {panelist} say about climate change?",
            f"What are {panelist}'s most controversial views?",
            f"How did {panelist} handle tough questions?",
        ])
    
    # Just topic questions  
    if topic:
        questions.extend([
            f"What did panelists say about {topic.lower()}?",
            f"What are different perspectives on {topic.lower()}?",
            f"Which panelists had the strongest views on {topic.lower()}?",
        ])
    
    # Always include general questions
    questions.extend([
        "What are the most controversial topics discussed?",
        "Which panelists had the most heated exchanges?",
        "What are the main political divides shown?",
        "What topics generated the most debate?",
    ])
    
    # Remove duplicates while preserving order
    seen = set()
    unique_questions = []
    for q in questions:
        if q not in seen:
            seen.add(q)
            unique_questions.append(q)  # Dataframe format
    
    return unique_questions[:10]  # Limit to 10 questions

def update_sample_questions(panelist, topic):
    new_questions = generate_sample_questions(panelist, topic)
    return gr.update(choices=new_questions, value=None)  # Radio needs 'choices' not 'value'



In [ ]:
# Create the Gradio interface
print("🎨 Creating Gradio interface...")

with gr.Blocks(
    # title="Q&A System V2 - Test",
    # theme=gr.themes.Soft()
    title="Q&A System V2 - Test",
    theme=gr.themes.Soft(),
    css="""
    .gr-radio-group {
        max-height: 200px;
        overflow-y: auto;
        border: 1px solid #e5e7eb;
        border-radius: 8px;
        padding: 12px;
        background: #fafafa;
    }
    .gr-radio-group label {
        padding: 4px 8px !important;
        margin: 2px 0 !important;
        border-radius: 4px;
    }
    .gr-radio-group label:hover {
        background: #f0f0f0;
    }
    #sample-questions {
        max-height: 200px !important;
        overflow-y: auto !important;
        border: 1px solid #ddd;
        border-radius: 6px;
        padding: 8px;
    }
    #question-container {
        position: relative !important;
    }
    #clear-btn-inside {
        position: absolute !important;
        top: 32px !important;
        right: 8px !important;
        width: 20px !important;
        height: 20px !important;
        min-width: 20px !important;
        padding: 0 !important;
        font-size: 12px !important;
        opacity: 0.5 !important;
        z-index: 10 !important;
        border-radius: 50% !important;
    }
    #clear-btn-inside:hover {
        opacity: 0.8 !important;
    }
  
    """    
) as demo:
    
    gr.Markdown("# 🧠 Q&A System V2 - Pipeline Test")
    gr.Markdown("Testing the refactored pipeline with basic Gradio interface.")
    
    with gr.Row():
        # Left column - Filtering
        with gr.Column(scale=1):
            gr.Markdown("### 1️⃣ Select Content")
            
            episode_dropdown = gr.Dropdown(
                label="Episode",
                choices=sorted(helpers.episode_lookup.keys()),
                value=sorted(helpers.episode_lookup.keys())[0],  # Set first episode as default
                interactive=True,
            )
            
            # And initialize the other dropdowns with that episode's data
            default_episode = sorted(helpers.episode_lookup.keys())[0]
            gr.Markdown("**Filter by:** *(based on selected episode)*")

            with gr.Row():
                with gr.Column():
                    panellist_radio = gr.Radio(
                        label="Panellist (in this episode)", 
                        choices=helpers.get_panellists_by_episode(default_episode),  # Pre-populate
                        interactive=True
                    )
                    # panellist_checkbox = gr.CheckboxGroup(
                    #     label="Panellist(s) (in this episode)", 
                    #     choices=helpers.get_panellists_by_episode(default_episode),
                    #     interactive=True
                    # )
                with gr.Column():
                    subtopic_radio = gr.Radio(
                        label="Topic (in this episode)", 
                        choices=helpers.get_subtopics_by_episode(default_episode),  # Pre-populate
                        interactive=True
                    )
        
        # Right column - Question input (keep your existing code)
        with gr.Column(scale=2):
            # Sample questions
            sample_questions = gr.Radio(
                label="Sample Questions (click to select)",
                choices=[
                    "What did panelists say about climate change?",
                    "How do panelists view immigration policy?", 
                    "What are different perspectives on the economy?",
                ],
                interactive=True,
                elem_id="sample-questions"
                #container=True,  # Shows as a container with scroll
            )   
            # Keep them in the same row but use CSS positioning
            with gr.Row():
                with gr.Column(scale=1, elem_id="question-container"):
                    question = gr.Textbox(
                        label="Your Question",
                        placeholder="e.g., What did panelists say about climate change?",
                        lines=3,
                        elem_id="question-input"
                    )
                    clear_question_btn = gr.Button(
                        "✕", 
                        variant="secondary", 
                        size="sm",
                        elem_id="clear-btn-inside"
                    )
            # Controls
            with gr.Row():
                k_slider = gr.Slider(
                    minimum=5,
                    maximum=50, 
                    value=15,  # Start small for testing
                    step=5,
                    label="Documents to retrieve (k)"
                )
                
                style_radio = gr.Radio(
                    label="Response Style",
                    choices=["Concise", "Balanced", "Detailed"],
                    value="Balanced"
                )
            
            # Buttons
            with gr.Row():
                submit_btn = gr.Button(
                    "🔍 Ask Question", 
                    variant="primary",
                    interactive=False  # Start disabled
                )   
                clear_btn = gr.Button("🗑️ Clear", variant="secondary")
        
        with gr.Column(scale=1):
            # Status
            gr.Markdown("### System Status")
            status_display = gr.Textbox(
                label="Status",
                value="Ready to test!",
                interactive=False
            )
            
            # Config info
            gr.Markdown(f"""
            **Configuration:**
            - Model: {config.chat_model_name}
            - Temperature: {config.temperature}
            - Debug: {config.debug_enabled}
            """)
    
    # Results
    gr.Markdown("### 📝 Response")
    answer_display = gr.Markdown(
        value="*Ask a question to see the response here.*"
    )
    
    gr.Markdown("### 📚 Sources") 
    sources_display = gr.Markdown(
        value="*Sources will appear here.*"
    )
    
    # Event handlers
    def set_sample_question(selected_question):
        return selected_question if selected_question else ""

    
    def clear_inputs():
        return "", "*Ask a question to see the response here.*", "Cleared - ready for next question"
    
    def update_panellists(ep_label):
        new_panelists = helpers.get_panellists_by_episode(ep_label)
        return gr.update(choices=new_panelists, value=None)  # Clear selection

    def update_subtopics(ep_label):
        new_subtopics = helpers.get_subtopics_by_episode(ep_label)
        return gr.update(choices=new_subtopics, value=None)  # Clear selection

    def filter_subtopics_by_panellist(panellist_name, ep_label):
        if panellist_name is None:
            return gr.update()
        
        # Get new subtopics and clear the selection
        new_subtopics = helpers.get_subtopics_by_panellist(ep_label, panellist_name)
        return gr.update(choices=new_subtopics, value=None)  # value=None clears selection

    def clear_question():
        return "", gr.update(interactive=False)  # Clear text + disable button

    def update_submit_button(question_text):
        # Enable button only if question has content
        has_content = bool(question_text.strip())
        return gr.update(interactive=has_content)
    

    # Wire up events
    episode_dropdown.change(
        fn=update_panellists,
        inputs=episode_dropdown,
        outputs=panellist_radio,
    )
    episode_dropdown.change(
        fn=update_subtopics,
        inputs=episode_dropdown,
        outputs=subtopic_radio,
    )
    panellist_radio.change(
        fn=filter_subtopics_by_panellist,
        inputs=[panellist_radio, episode_dropdown],
        outputs=subtopic_radio,
    )    
    sample_questions.change(
        fn=set_sample_question,
        inputs=[sample_questions],
        outputs=[question]
    )
        # Add these event handlers
    panellist_radio.change(
        fn=update_sample_questions,
        inputs=[panellist_radio, subtopic_radio],
        outputs=sample_questions
    )

    subtopic_radio.change(
        fn=update_sample_questions,
        inputs=[panellist_radio, subtopic_radio], 
        outputs=sample_questions
    )
    submit_btn.click(
        fn=handle_question,
        inputs=[question, k_slider, style_radio],
        outputs=[answer_display, sources_display, status_display]
    )
    
    clear_btn.click(
        fn=clear_inputs,
        inputs=[],
        outputs=[question, answer_display, status_display]
    )

    clear_question_btn.click(
        fn=clear_question,
        inputs=[],
        outputs=[question, submit_btn]
    )

    question.change(
        fn=update_submit_button,
        inputs=[question],
        outputs=[submit_btn]
    )

print("✅ Gradio interface created!")

In [ ]:
# Launch the interface
print("🌐 Launching Gradio interface...")

demo.launch(
    share=True,  # Set to True for public link
    server_port=7860,
    show_error=True,
    debug=True  # Shows more info in console
)

In [ ]:
# If you need to stop the demo
demo.close()

## 🧪 Quick Tests

Use the cells below for rapid testing and debugging:

In [ ]:
# Quick test different parameters
test_questions = [
    "What did panelists say about climate change?",
    "How do panelists view immigration?",
    "What are the different economic perspectives?"
]

for i, q in enumerate(test_questions[:1]):  # Test just first one
    print(f"\n=== Test {i+1}: {q} ===")
    result = handle_question(q, 10, "Concise")
    print(f"Status: {result[2]}")
    print(f"Answer (first 100 chars): {result[0][:100]}...")

In [ ]:
# Debug: Check what's in your helpers object
print("Helpers object attributes:")
print([attr for attr in dir(helpers) if not attr.startswith('_')])

print(f"\nDatabase info:")
print(f"- Episodes: {len(helpers.df_ep)}")
print(f"- Responses: {len(helpers.df_reply)}")
print(f"- Panelists: {len(helpers.df_guests)}")